# HoneyBee YOLO-Pose training 

Разделы: 
- предоббработка данных 
- обучение YOLO-Pose 11 Nano

In [ ]:
import sys
print(sys.executable)

C:\Users\Asya\AppData\Local\Programs\Python\Python313\python.exe


In [ ]:
import sys

!{sys.executable} -m pip install opencv-python ultralytics supervision pyyaml tqdm pandas matplotlib pillow

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ----------------------- ---------------- 0.8/1.3 MB 5.2 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 5.1 MB/s  0:00:00
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ------ --------------------------------- 1.6/9.3 MB 9.3 MB/s eta 0:00:01
   ---------------- ----------------------- 3.9/9.3 MB 9.9


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# загрузка библиотек
import sys
import os
from pathlib import Path
import random, shutil, math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import yaml

import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch, Circle
from PIL import Image

from tqdm.auto import tqdm
import torch
from ultralytics import YOLO

#import supervision as sv
#from trackers import OCSORTTracker

%matplotlib inline

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

C:\Users\Asya\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: 0


In [ ]:
# config
PROJECTDIR = Path(r"C:\bee_yolo_project")

RAWDIR = PROJECTDIR / "raw_kaggle"

FRAMESDIR = RAWDIR / "frames"
ANNOTATIONSDIR = RAWDIR / "positions"

YOLODATASETDIR = PROJECTDIR / "yolo-bees-pose-dataset"
RUNSDIR = PROJECTDIR / "runs"
TRACKOUTPUTDIR = PROJECTDIR / "tracking-outputs"
EXPORTDIR = PROJECTDIR / "exports"

for d in [YOLODATASETDIR, RUNSDIR, TRACKOUTPUTDIR, EXPORTDIR]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAINRATIO = 0.70
VALRATIO = 0.15
TESTRATIO = 0.15
SOURCECROPSIZE = 512
YOLOIMGSZ = 640

RUNNAME = "bee_pose_yolo11s_640_80ep"
DATAYAML = YOLODATASETDIR / "data.yaml"

FULLBOXLENGTHSCALE = 1.95
FULLBBOXWIDTHSCALE = 1.15
PARTIALBBOXSCALE = 1.15
BODYAXISOFFSETRAD = math.pi / 2
HEADTAILFLIPRAD = math.pi

random.seed(SEED)
np.random.seed(SEED)

print("PROJECTDIR:", PROJECTDIR)
print("FRAMESDIR:", FRAMESDIR, FRAMESDIR.exists())
print("ANNOTATIONSDIR:", ANNOTATIONSDIR, ANNOTATIONSDIR.exists())
print("YOLODATASETDIR:", YOLODATASETDIR)
print("DEVICE:", DEVICE)

PROJECTDIR: C:\bee_yolo_project
FRAMESDIR: C:\bee_yolo_project\raw_kaggle\frames True
ANNOTATIONSDIR: C:\bee_yolo_project\raw_kaggle\positions True
YOLODATASETDIR: C:\bee_yolo_project\yolo-bees-pose-dataset
DEVICE: 0


In [ ]:
import torch

print("torch:", torch.__version__)
print("torch cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.11.0+cu128
torch cuda version: 12.8
cuda available: True
gpu: NVIDIA GeForce RTX 5070 Ti


In [ ]:
import sys

!{sys.executable} -m pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.26.0
Uninstalling torchvision-0.26.0:
  Successfully uninstalled torchvision-0.26.0


You can safely remove it manually.
You can safely remove it manually.


In [ ]:
import sys

!{sys.executable} -m pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-win_amd64.whl.metadata (5.6 kB)
  Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
Using cached https://download-r2.pytorch.org/whl/cu128/torch-2.11.0%2Bcu128-cp313-cp313-win_amd64.whl (2753.2 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchvision-0.26.0%2Bcu128-cp313-cp313-win_amd64.whl (9.6 MB)
Using cached https://download-r2.pytorch.org/whl/cu128/torchaudio-2.11.0%2Bcu128-cp313-cp313-win_amd64.whl (1.7 MB)

   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ---------------------------------------- 0/3 [torchaudio]
   ------------- -----------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Предобработка и подготовка данных


In [ ]:
def findpairs(framesdir: Path, annotationsdir: Path):
    pairs = []
    for ext in (".png", ".jpg", ".jpeg", ".tif", ".tiff"):
        for imgpath in framesdir.glob(f"*{ext}"):
            annpath = annotationsdir / f"{imgpath.stem}.txt"
            if annpath.exists():
                pairs.append((imgpath, annpath))
    pairs = sorted(pairs)
    if not pairs:
        raise FileNotFoundError(f"No image/txt pairs found in {framesdir} and {annotationsdir}")
    return pairs

def readannotation(annpath: Path):
    ann = pd.read_csv(
        annpath,
        sep=r"\s+",
        header=None,
        names=["offsetx", "offsety", "sourceclass", "x", "y", "angle"],
        engine="python",
    )
    for c in ann.columns:
        ann[c] = pd.to_numeric(ann[c], errors="coerce")
    ann = ann.dropna().copy()
    ann[["offsetx", "offsety", "sourceclass"]] = ann[["offsetx", "offsety", "sourceclass"]].astype(int)
    return ann

def clipvalue(value, lo, hi):
    return max(lo, min(hi, value))

def pointinsidecrop(x, y, cropsize=SOURCECROPSIZE):
    return 0 <= x < cropsize and 0 <= y < cropsize

def bboxsizeforsourceclass(sourceclass):
    if int(sourceclass) == 2:
        size = 2 * 20.0 * PARTIALBBOXSCALE
        return size, size
    return 2 * 35.0 * FULLBOXLENGTHSCALE, 2 * 20.0 * FULLBBOXWIDTHSCALE

def orientationtheta(angle):
    return float(angle + BODYAXISOFFSETRAD + HEADTAILFLIPRAD)

def bboxfromsourcegeometry(cx, cy, angle, sourceclass):
    length, width = bboxsizeforsourceclass(sourceclass)
    if int(sourceclass) == 2:
        half = length / 2
        return cx - half, cy - half, cx + half, cy + half
    theta = orientationtheta(angle)
    ux, uy = math.cos(theta), math.sin(theta)
    px, py = -uy, ux
    halfl, halfw = length / 2, width / 2
    corners = []
    for sl in (-1, 1):
        for sw in (-1, 1):
            corners.append(
                (
                    cx + ux * halfl * sl + px * halfw * sw,
                    cy + uy * halfl * sl + py * halfw * sw,
                )
            )
    xs = [p[0] for p in corners]
    ys = [p[1] for p in corners]
    return min(xs), min(ys), max(xs), max(ys)

def headtailfromcenterangle(cx, cy, angle, sourceclass=1):
    if int(sourceclass) == 2:
        return (cx, cy), (cx, cy)
    length = 2 * 35.0 * FULLBOXLENGTHSCALE
    theta = orientationtheta(angle)
    ux, uy = math.cos(theta), math.sin(theta)
    halfl = length / 2
    return (cx + ux * halfl, cy + uy * halfl), (cx - ux * halfl, cy - uy * halfl)

def kptpt(pt, visibledefault=2):
    x, y = pt
    if visibledefault == 0:
        return 0.0, 0.0, 0
    if pointinsidecrop(x, y):
        return x, y, visibledefault
    return 0.0, 0.0, 0

def yoloposelinefromsourcerow(row, cropsize=SOURCECROPSIZE):
    cx, cy, sourceclass, angle = float(row.x), float(row.y), int(row.sourceclass), float(row.angle)
    if not pointinsidecrop(cx, cy, cropsize):
        return None

    x1, y1, x2, y2 = bboxfromsourcegeometry(cx, cy, angle, sourceclass)
    x1, y1, x2, y2 = (
        clipvalue(x1, 0, cropsize),
        clipvalue(y1, 0, cropsize),
        clipvalue(x2, 0, cropsize),
        clipvalue(y2, 0, cropsize),
    )

    bw, bh = max(1.0, x2 - x1), max(1.0, y2 - y1)
    bx, by = (x1 + x2) / 2, (y1 + y2) / 2

    head, tail = headtailfromcenterangle(cx, cy, angle, sourceclass)
    hx, hy, hv = kptpt(head, 2)
    tx, ty, tv = kptpt(tail, 2)

    class_id = 0  # один класс bee

    vals = [
        class_id,
        bx / cropsize, by / cropsize, bw / cropsize, bh / cropsize,
        hx / cropsize, hy / cropsize, hv,
        tx / cropsize, ty / cropsize, tv,
    ]
    return " ".join(str(v) if isinstance(v, int) else f"{v:.6f}" for v in vals)

In [ ]:
# конвертер датасета
@dataclass
class OriginalGeometryPoseConverter:
    framesdir: Path
    annotationsdir: Path
    outputdir: Path
    sourcecropsize: int = SOURCECROPSIZE
    seed: int = SEED
    keepemptycrops: bool = False

    def preparedirs(self):
        if self.outputdir.exists():
            shutil.rmtree(self.outputdir)
        for split in ("train", "val", "test"):
            (self.outputdir / "images" / split).mkdir(parents=True, exist_ok=True)
            (self.outputdir / "labels" / split).mkdir(parents=True, exist_ok=True)

    def splitpairs(self, pairs):
        rng = random.Random(self.seed)
        pairs = list(pairs)
        rng.shuffle(pairs)
        n = len(pairs)
        trainend = int(n * TRAINRATIO)
        valend = int(n * (TRAINRATIO + VALRATIO))
        return {
            "train": pairs[:trainend],
            "val": pairs[trainend:valend],
            "test": pairs[valend:],
        }

    def processpair(self, imgpath: Path, annpath: Path, split: str):
        img = Image.open(imgpath).convert("RGB")
        ann = readannotation(annpath)
        cropscreated = objectscreated = skippedoob = 0

        for (offsetx, offsety), cropann in ann.groupby(["offsetx", "offsety"]):
            crop = img.crop(
                (offsetx, offsety, offsetx + self.sourcecropsize, offsety + self.sourcecropsize)
            ).convert("RGB")
            if crop.size != (self.sourcecropsize, self.sourcecropsize):
                continue

            lines = []
            for row in cropann.itertuples(index=False):
                line = yoloposelinefromsourcerow(row, self.sourcecropsize)
                if line is None:
                    skippedoob += 1
                    continue
                lines.append(line)

            if not lines and not self.keepemptycrops:
                continue

            cropname = f"{imgpath.stem}_x{offsetx}_y{offsety}"
            crop.save(self.outputdir / "images" / split / f"{cropname}.png")
            (self.outputdir / "labels" / split / f"{cropname}.txt").write_text("\n".join(lines))
            cropscreated += 1
            objectscreated += len(lines)

        return cropscreated, objectscreated, skippedoob

    def writeyaml(self):
        data = {
            "path": str(self.outputdir),
            "train": "images/train",
            "val": "images/val",
            "test": "images/test",
            "kpt_shape": [2, 3],
            "names": {0: "bee"},
        }
        with open(self.outputdir / "data.yaml", "w", encoding="utf-8") as f:
            yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

    def convert(self, maxframes=None):
        pairs = findpairs(self.framesdir, self.annotationsdir)
        if maxframes:
            pairs = pairs[:maxframes]

        self.preparedirs()
        summary = {}

        for split, splitpairs in self.splitpairs(pairs).items():
            crops = objects = skipped = 0
            for imgpath, annpath in tqdm(splitpairs, desc=f"convert {split}"):
                c, o, s = self.processpair(imgpath, annpath, split)
                crops += c
                objects += o
                skipped += s
            summary[split] = {"crops": crops, "objects": objects, "skipped": skipped}

        self.writeyaml()
        return summary

In [ ]:
# подготовка данных
PREPAREDATASET = True
MAXFRAMES = None

if PREPAREDATASET:
    converter = OriginalGeometryPoseConverter(FRAMESDIR, ANNOTATIONSDIR, YOLODATASETDIR)
    summary = converter.convert(maxframes=MAXFRAMES)
    print(summary)
    print(DATAYAML)

convert test: 100%|██████████████████████████████████████████████████████████████████| 108/108 [01:16<00:00,  1.41it/s]

{'train': {'crops': 15096, 'objects': 252624, 'skipped': 10165}, 'val': {'crops': 3312, 'objects': 55084, 'skipped': 2187}, 'test': {'crops': 3192, 'objects': 53527, 'skipped': 2111}}
C:\bee_yolo_project\yolo-bees-pose-dataset\data.yaml


In [ ]:
# проверка конвертера
def count_files(folder, exts=(".png", ".jpg", ".jpeg", ".tif", ".tiff")):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return sum(1 for p in folder.iterdir() if p.suffix.lower() in exts)

def count_labels(folder):
    folder = Path(folder)
    if not folder.exists():
        return 0
    return len(list(folder.glob("*.txt")))

def count_objects(label_dir):
    n = 0
    for f in Path(label_dir).glob("*.txt"):
        txt = f.read_text(encoding="utf-8").strip()
        if txt:
            n += len(txt.splitlines())
    return n

def dataset_summary_df(dataset_dir):
    rows = []
    for split in ["train", "val", "test"]:
        img_dir = Path(dataset_dir) / "images" / split
        lab_dir = Path(dataset_dir) / "labels" / split
        n_images = count_files(img_dir)
        n_labels = count_labels(lab_dir)
        n_objects = count_objects(lab_dir)
        rows.append({
            "split": split,
            "images": n_images,
            "labels": n_labels,
            "objects": n_objects,
            "objects_per_image": round(n_objects / max(n_images, 1), 3),
        })
    return pd.DataFrame(rows)

ds_df = dataset_summary_df(YOLODATASETDIR)
print("СВОДКА ПО ДАТАСЕТУ")
display(ds_df)

print("ПРОВЕРКА СТРУКТУРЫ")
display(pd.DataFrame({
    "path": [str(YOLODATASETDIR / "images" / s) for s in ["train", "val", "test"]],
    "exists": [(YOLODATASETDIR / "images" / s).exists() for s in ["train", "val", "test"]],
}))

СВОДКА ПО ДАТАСЕТУ


,split,images,labels,objects,objects_per_image
0,train,15096,15096,252624,16.734
1,val,3312,3312,55084,16.632
2,test,3192,3192,53527,16.769


ПРОВЕРКА СТРУКТУРЫ


,path,exists
0,C:\bee_yolo_project\yolo-bees-pose-dataset\ima...,True
1,C:\bee_yolo_project\yolo-bees-pose-dataset\ima...,True
2,C:\bee_yolo_project\yolo-bees-pose-dataset\ima...,True


In [ ]:
# проверка labels
label_file = next((YOLODATASETDIR / "labels" / "train").glob("*.txt"))
print(label_file.read_text())

0 0.421875 0.347656 0.121540 0.275536 0.438055 0.479971 2 0.405695 0.215341 2
0 0.136719 0.406250 0.205875 0.277237 0.200145 0.289006 2 0.073292 0.523494 2
0 0.468750 0.710938 0.162569 0.281171 0.507002 0.583243 2 0.430498 0.838632 2
0 0.195312 0.859375 0.089844 0.266602 0.195312 0.726074 2 0.195313 0.992676 2
0 0.417969 0.136719 0.089844 0.089844 0.417969 0.136719 2 0.417969 0.136719 2
0 0.804688 0.160156 0.270253 0.101340 0.937862 0.165947 2 0.671513 0.154366 2
0 0.878906 0.464844 0.145888 0.279786 0.908012 0.594928 2 0.849800 0.334759 2
0 0.911922 0.926056 0.176156 0.147888 0.000000 0.000000 0 0.000000 0.000000 0


## Обучение YOLO-Pose

In [ ]:
model = YOLO("yolo11s-pose.pt")

results = model.train(
    data=str(DATAYAML),
    task="pose",
    epochs=80,
    imgsz=YOLOIMGSZ,
    batch=16,
    workers=4,
    patience=20,
    device=DEVICE,
    fliplr=0.0,
    mosaic=1.0,
    amp=True,
    cache=False,
    save=True,
    save_period=10,
    plots=True,
    project=str(RUNSDIR),
    name=RUNNAME,
    exist_ok=True,
)

print(results)

Ultralytics 8.4.71  Python-3.13.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\bee_yolo_project\yolo-bees-pose-dataset\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=bee_pose_yolo11s_640_80ep, nbs=64, nms=False, opset=None, opt